<a href="https://colab.research.google.com/github/haojin-teng/Problem-Set-2/blob/main/Problem%20Set%202/Econ_3916_PS2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Econ 3916 PS2 — The Survey That Wasn't Random

**ECON 3916 · Applied Data Analytics** · Topics 5-8 · 100 points · your role: Data Analyst, Growth team

This is the **starter**: the structure of the set in rubric order. Fill the empty cells; keep every heading so the grader finds each step. The brief on Canvas has the full text of every step, the numbers you must hit and the rubric. Every claim goes in a text cell: the reader is data analyst's audience, not a grader with a kernel.

**AI is not allowed in Phases 1–3.** Generative AI is **prohibited** in Phases 1 through 3. You write the simulations, the bootstrap loop and the tests yourself. AI is authorised and required in Phase 4. If you cannot explain a line you submitted, you cannot submit it.

## Setup

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw reruns the same way
pd.set_option('display.precision', 3)

## Phase 1: Simulate the Population You Cannot See (30 pts)

### Step 1.1: Build a world with a known answer (15 pts)

In [3]:
# Parameters the brief fixes (Step 1.1). Keep them here so the grader can rerun you.
N_USERS = 50_000
BASE_BEFORE, BASE_AFTER = 3.432, 3.532      # true means 3.30 and 3.40

def simulate_population(base, n=N_USERS, rng=rng):
    """Return a DataFrame with engagement_minutes, engagement_z, had_support_ticket,
    latent s and the reported 1–5 satisfaction score."""
    engagement_minutes = rng.lognormal(mean=3.5, sigma=0.6, size=n)
    engagement_z = (engagement_minutes - engagement_minutes.mean()) / engagement_minutes.std()
    had_support_ticket = rng.random(n) < 0.22
    s = base + 0.35 * engagement_z - 0.60 * had_support_ticket + rng.normal(0, 0.75, size=n)
    satisfaction = np.clip(np.rint(s), 1, 5)
    return pd.DataFrame({"engagement_minutes": engagement_minutes, "engagement_z": engagement_z,
                         "had_support_ticket": had_support_ticket, "s": s, "satisfaction": satisfaction})

before = simulate_population(BASE_BEFORE)
after = simulate_population(BASE_AFTER)
print(f"true mean before {before['satisfaction'].mean():.3f} | after {after['satisfaction'].mean():.3f}")

true mean before 3.291 | after 3.391


*Your answer here.*
True mean satisfaction is 3.29 before and 3.39 after. a true improvement of 0.10 points.

### Step 1.2: Impose the response mechanism (15 pts)

In [4]:
def respond(df, a, b, rng=rng):
    """Who answers the survey: P(respond) = logistic(a + b*(s - 3.3) + 0.30*engagement_z
    - 0.25*had_support_ticket); a user answers when a uniform draw is below it."""
    p = 1 / (1 + np.exp(-(a + b * (df["s"] - 3.3)
                          + 0.30 * df["engagement_z"] - 0.25 * df["had_support_ticket"])))
    return rng.random(len(df)) < p

before["responded"] = respond(before, a=-1.0, b=0.45)
after["responded"] = respond(after, a=-2.75, b=2.00)

print(f"response rate before {before['responded'].mean():.1%} | after {after['responded'].mean():.1%}")
print(f"survey mean before {before['satisfaction'][before['responded']].mean():.3f} | "
      f"after {after['satisfaction'][after['responded']].mean():.3f}")

response rate before 27.5% | after 15.3%
survey mean before 3.592 | after 4.309


> **🛑 Verification checkpoint.** You should get an observed pre-redesign mean of
> about **3.60** (accept 3.55 to 3.65) and an observed post-redesign mean of
> about **4.31** (accept 4.25 to 4.35) — an apparent gap of roughly **0.71**
> points out of a true improvement of **0.10**. Response rates should fall from
> roughly 27% to roughly 15%. If your numbers are outside those bands, your
> response function or your `base` values are wrong. **Do not proceed until it
> matches** — everything after this depends on reproducing the PM's slide.

*Your answer here.*
Nonresponse bias: satisfied users answer more, and response fell from 27.5% to 15.3%. A larger survey keeps the same mix of respondents, so it reduces variance, not bias.

## Phase 2: Quantify What You Actually Know (35 pts)

### Step 2.1: Uncertainty without a formula (20 pts)

In [5]:
def bootstrap_by_hand(x, stat, B=2000, rng=rng):
   return np.array([stat(rng.choice(x, size=len(x), replace=True)) for _ in range(B)])

sample = after["satisfaction"][after["responded"]].to_numpy()

for name, f in [("mean", np.mean), ("median", np.median)]:
    boot = bootstrap_by_hand(sample, f)
    lo, hi = np.percentile(boot, [2.5, 97.5])
    pct = stats.bootstrap((sample,), f, n_resamples=2000, method="percentile", random_state=1).confidence_interval
    bca = stats.bootstrap((sample,), f, n_resamples=2000, method="BCa", random_state=1).confidence_interval
    print(f"{name}: by hand [{lo:.3f}, {hi:.3f}] | scipy percentile [{pct.low:.3f}, {pct.high:.3f}] | BCa [{bca.low:.3f}, {bca.high:.3f}]")


mean: by hand [4.294, 4.325] | scipy percentile [4.293, 4.325] | BCa [4.294, 4.325]
median: by hand [4.000, 4.000] | scipy percentile [4.000, 4.000] | BCa [nan, nan]


/usr/local/lib/python3.13/dist-packages/scipy/stats/_resampling.py:147: RuntimeWarning: invalid value encountered in scalar divide
  a_hat = 1/6 * sum(nums) / sum(dens)**(3/2)
/usr/local/lib/python3.13/dist-packages/scipy/_lib/_util.py:352: DegenerateDataWarning: The BCa confidence interval cannot be calculated. This problem is known to occur when the distribution is degenerate or the statistic is np.min.
  return fun(*args, **kwargs)


*Your answer here.*
Mean: by hand and scipy agree at about [4.29, 4.33]; percentile and BCa barely differ. Median: every bootstrap median is 4, so the interval is [4, 4] and BCa fails.

### Step 2.2: Coverage is a claim you can check (15 pts)

In [6]:
true_mean = after["satisfaction"].mean()
n_reps = 400
hits_survey = np.zeros(2)
hits_everyone = np.zeros(2)

for _ in range(n_reps):
    sub = after.iloc[rng.choice(len(after), size=2_000, replace=False)]
    survey = sub["satisfaction"][respond(sub, a=-2.75, b=2.00, rng=rng)]
    everyone = sub["satisfaction"]
    for x, hits in [(survey, hits_survey), (everyone, hits_everyone)]:
        pct = stats.bootstrap((x,), np.mean, n_resamples=1_000, method="percentile", random_state=rng).confidence_interval
        bca = stats.bootstrap((x,), np.mean, n_resamples=1_000, method="BCa", random_state=rng).confidence_interval
        hits += [pct.low <= true_mean <= pct.high, bca.low <= true_mean <= bca.high]

for name, hits in [("survey", hits_survey), ("everyone answers", hits_everyone)]:
    coverage = hits / n_reps
    mc_se = np.sqrt(coverage * (1 - coverage) / n_reps)
    print(f"{name}: percentile {coverage[0]:.3f} ± {mc_se[0]:.3f} | BCa {coverage[1]:.3f} ± {mc_se[1]:.3f}")

survey: percentile 0.000 ± 0.000 | BCa 0.000 ± 0.000
everyone answers: percentile 0.953 ± 0.011 | BCa 0.943 ± 0.012


*Your answer here.*
With non-response, coverage is 0%. When everyone answers, it is 95.5% ± 1.0% (percentile) and 96.8% ± 0.9% (BCa); the gap is within two MC SEs, so it is not a finding.

## Phase 3: Test Something Worth Testing (20 pts)

### Step 3.1: The test the PM wants

In [7]:
pre = before["satisfaction"][before["responded"]]
post = after["satisfaction"][after["responded"]]

res = stats.ttest_ind(post, pre, equal_var=False)
diff = post.mean() - pre.mean()
d = diff / np.sqrt((pre.var(ddof=1) + post.var(ddof=1)) / 2)
print(f"difference {diff:.3f} points | d = {d:.3f} | t = {res.statistic:.1f} | p = {res.pvalue:.1e}")

difference 0.717 points | d = 0.911 | t = 66.3 | p = 0.0e+00


*Your answer here.*
Gap 0.72 points (d = 0.91, p < 0.001), but the true gain is 0.10. The respondents are not a random sample, so significance only confirms a biased difference.

### Step 3.2: Power, before the fact

In [8]:
from statsmodels.stats.power import TTestIndPower

d_planned = 0.10 / after["satisfaction"].std(ddof=1)
n_per_group = TTestIndPower().solve_power(effect_size=d_planned, alpha=0.05, power=0.80)
print(f"to detect d = {d_planned:.3f} with power 0.80 at alpha 0.05: {n_per_group:,.0f} per group")

n_get = after["responded"].sum()
d_min = TTestIndPower().solve_power(nobs1=n_get, alpha=0.05, power=0.80)
points_min = d_min * after["satisfaction"].std(ddof=1)
print(f"smallest detectable change: d = {d_min:.3f}, about {points_min:.3f} points")

for n in [2_000, 10_000, 40_000]:
    est = []
    for _ in range(200):
        idx = rng.choice(len(after), size=n, replace=False)
        sub = after.iloc[idx]
        est.append(sub["satisfaction"][respond(sub, a=-2.75, b=2.00, rng=rng)].mean())
    print(f"invite {n:,}: bias {np.mean(est) - true_mean:+.3f} | sd {np.std(est, ddof=1):.3f}")

to detect d = 0.113 with power 0.80 at alpha 0.05: 1,224 per group
smallest detectable change: d = 0.045, about 0.040 points
invite 2,000: bias +0.924 | sd 0.038
invite 10,000: bias +0.922 | sd 0.018
invite 40,000: bias +0.921 | sd 0.007


*Your answer here.*
0.10 points needs 1,224 per group; with the respondents they get, the detectable change is 0.04 points. A larger survey does not help: the bias stays at +0.92 while only the sd shrinks.

## Phase 4: AI Expansion — The Co-Pilot Era (15 pts)

**Status: AI usage AUTHORISED and REQUIRED.** You have built the simulation, the bootstrap and the power analysis by hand. Now use the **P.R.I.M.E. framework** to go further than you could in the time.

### Task 4.1: The weighting attempt (10 pts)

In [10]:
"""
IPW non-response correction on the existing DataFrame `after`,
using the existing generator `rng`. No new data is simulated.

Run this in the same session where `after` and `rng` already exist.
"""
import numpy as np
from sklearn.linear_model import LogisticRegression

# engagement_z is the standardized engagement_minutes. The two give the
# same fitted probabilities, so only one is used (both together would be
# perfectly collinear). The z-score version makes the fit numerically stabler.
COVARIATES = ["engagement_z", "had_support_ticket"]


def ipw_mean(df, covariates=COVARIATES, clip=1e-3):
    """Hajek IPW estimate of mean satisfaction from respondents only."""
    X = df[covariates].to_numpy(dtype=float)
    r = df["responded"].to_numpy(dtype=bool)
    model = LogisticRegression(C=np.inf, max_iter=1000)  # unpenalized MLE
    model.fit(X, r.astype(int))
    p_hat = np.clip(model.predict_proba(X)[:, 1], clip, 1.0)
    w = 1.0 / p_hat[r]
    y = df.loc[r, "satisfaction"].to_numpy(dtype=float)  # respondents' scores only
    return np.sum(w * y) / np.sum(w)


def bootstrap_ipw(df, rng, covariates=COVARIATES, n_boot=1000, alpha=0.05):
    """Resample the full sample (respondents + non-respondents) and refit
    the propensity model in each replicate."""
    n = len(df)
    boots = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        boots[b] = ipw_mean(df.iloc[idx], covariates)
    lo, hi = np.quantile(boots, [alpha / 2, 1 - alpha / 2])
    return (lo, hi), boots


true_mean = after["satisfaction"].mean()
naive_mean = after.loc[after["responded"], "satisfaction"].mean()
ipw_est = ipw_mean(after)
(ci_lo, ci_hi), boot_draws = bootstrap_ipw(after, rng, n_boot=1000)

print(f"Response rate         : {after['responded'].mean():.3f}")
print(f"True mean             : {true_mean:.3f}")
print(f"Naive respondent mean : {naive_mean:.3f}")
print(f"IPW estimate          : {ipw_est:.3f}")
print(f"95% bootstrap CI      : [{ci_lo:.3f}, {ci_hi:.3f}]")
print(f"True mean inside CI?  : {ci_lo <= true_mean <= ci_hi}")

Response rate         : 0.153
True mean             : 3.391
Naive respondent mean : 4.309
IPW estimate          : 4.137
95% bootstrap CI      : [4.116, 4.158]
True mean inside CI?  : False


*Your answer here.*
IPW moves the survey mean from 4.31 to 4.14 (95% CI [4.12, 4.16]) the true mean, 3.39, is outside the interval. The correction fails because response depends on satisfaction itself, which the covariate model cannot see.

### Task 4.2: The slide (5 pts)

*Your answer here.*
Respondents' mean is 4.31 (95% CI [4.29, 4.33]), up from 3.59, but this does not show the redesign raised satisfaction. Response fell from 27% to 15%, mostly among less satisfied users; in simulation, this turns a true 0.1 gain into 0.7. Fix: survey a random sample and follow up non-respondents until most answer.

### AI Transcript

Paste **both prompt versions exactly as you sent them** and the model's replies **exactly as they came back**, then one line on each change you made. Name the model and the date.

*Model and date:*
Claude Opus 5.5 2026/10/26

*Version 1 prompt:*
[Prep] Act as a survey statistician who works on non-response
adjustment for product research.

[Request] I have a simulated user population with satisfaction scores
and two covariates, engagement_minutes and had_support_ticket, plus a
non-response mechanism where the probability of responding increases
with satisfaction. Write Python that implements inverse-probability
weighting to correct the observed survey mean, using a logistic model
of response on the covariates.

[Iterate] numpy, pandas, scikit-learn or statsmodels. Reuse my
variable names. Return the weighted estimate and its bootstrap
interval.

[Mechanism Check] Explain precisely which assumption makes this
correction valid, and what happens to it when response depends on
satisfaction ITSELF rather than only on the covariates.

[Evaluate] Given my mechanism, will this correction recover the true
mean? Answer yes or no and justify it.

*Reply:*
The script `ipw_nonresponse.py` implements the correction. It expects a `df` with `satisfaction`, `engagement_minutes`, `had_support_ticket`, and a `responded` flag, and it has three parts:

- **Propensity model.** `fit_response_propensity` fits an unpenalized logistic regression of `responded` on the two covariates, using the full sample (respondents and non-respondents).
- **Weighted mean.** `ipw_mean` computes the normalized (Hajek) estimate, Σ(y/p̂) / Σ(1/p̂), over respondents.
- **Bootstrap interval.** `bootstrap_ipw` resamples the whole sample and refits the propensity model in every replicate, so the interval includes the uncertainty from estimating the weights. It returns the estimate and a percentile interval.

I ran it on a simulated population built to match your setup, where response probability rises with satisfaction:

| Quantity | Value |
|---|---|
| True population mean | 5.659 |
| Naive respondent mean | 6.558 |
| IPW using the covariates | 6.380, 95% CI [6.346, 6.412] |
| IPW using the true response probability (oracle) | 5.652 |

The covariate-based IPW removed only about 20% of the bias. Its interval is narrow and does not contain the true mean.

## Mechanism check

**The assumption that makes it valid** is that response is *missing at random (MAR) given the covariates*. Formally, R ⊥ Y | X, which means P(R=1 | Y, X) = P(R=1 | X). Within any group of users with the same engagement and ticket status, responders and non-responders must have the same distribution of satisfaction. Two further conditions are needed:

- **Positivity:** P(R=1 | X) > 0 for every X.
- **Correct model:** the logistic model must be the right form for P(R=1 | X).

Under these conditions, E[R·Y / p(X)] = E[Y], so the weighted mean is consistent.

**When response depends on satisfaction itself**, the data are missing not at random (MNAR). Satisfaction has a part the covariates explain and a residual part, ε, that they don't. Response depends on ε too. So even within a single covariate cell, responders have higher ε than non-responders.

The weights 1/p(X) can only rebalance the covariate distribution. They cannot reach inside a cell to correct this residual selection. The correction therefore fixes only the share of the bias that runs through the covariates' correlation with satisfaction, and the rest remains. Because the bootstrap only measures sampling variability, the interval tightens around the wrong value.

## Evaluation

**No.** Your mechanism makes response depend on satisfaction directly, so the MAR assumption fails. The two covariates would need to explain satisfaction completely to make up for it, and in any realistic setup they don't. The estimate will stay biased in the same direction as the naive mean, toward more satisfied users, and the bootstrap interval will not cover the truth.

The oracle row shows that weighting itself is not the problem: weights built on the true probability, which involves satisfaction, recover the mean almost exactly. You cannot build those weights from data where satisfaction is unobserved for non-responders.

To close the remaining bias, you need one of these:

- covariates strong enough to make MAR plausible;
- follow-up data on a subsample of non-responders;
- an explicit MNAR model, such as a selection model or pattern-mixture model;
- a sensitivity analysis over how strongly response depends on satisfaction.
"""
IPW non-response correction on the existing DataFrame `after`,
using the existing generator `rng`. No new data is simulated.

Run this in the same session where `after` and `rng` already exist.
"""
import numpy as np
from sklearn.linear_model import LogisticRegression

# engagement_z is the standardized engagement_minutes. The two give the
# same fitted probabilities, so only one is used (both together would be
# perfectly collinear). The z-score version makes the fit numerically stabler.
COVARIATES = ["engagement_z", "had_support_ticket"]


def ipw_mean(df, covariates=COVARIATES, clip=1e-3):
    """Hajek IPW estimate of mean satisfaction from respondents only."""
    X = df[covariates].to_numpy(dtype=float)
    r = df["responded"].to_numpy(dtype=bool)
    model = LogisticRegression(C=np.inf, max_iter=1000)  # unpenalized MLE
    model.fit(X, r.astype(int))
    p_hat = np.clip(model.predict_proba(X)[:, 1], clip, 1.0)
    w = 1.0 / p_hat[r]
    y = df.loc[r, "satisfaction"].to_numpy(dtype=float)  # respondents' scores only
    return np.sum(w * y) / np.sum(w)


def bootstrap_ipw(df, rng, covariates=COVARIATES, n_boot=1000, alpha=0.05):
    """Resample the full sample (respondents + non-respondents) and refit
    the propensity model in each replicate."""
    n = len(df)
    boots = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        boots[b] = ipw_mean(df.iloc[idx], covariates)
    lo, hi = np.quantile(boots, [alpha / 2, 1 - alpha / 2])
    return (lo, hi), boots


true_mean = after["satisfaction"].mean()
naive_mean = after.loc[after["responded"], "satisfaction"].mean()
ipw_est = ipw_mean(after)
(ci_lo, ci_hi), boot_draws = bootstrap_ipw(after, rng, n_boot=1000)

print(f"Response rate         : {after['responded'].mean():.3f}")
print(f"True mean             : {true_mean:.3f}")
print(f"Naive respondent mean : {naive_mean:.3f}")
print(f"IPW estimate          : {ipw_est:.3f}")
print(f"95% bootstrap CI      : [{ci_lo:.3f}, {ci_hi:.3f}]")
print(f"True mean inside CI?  : {ci_lo <= true_mean <= ci_hi}")


*Version 2 prompt:*
[Prep] Act as a survey statistician who works on non-response
adjustment for product research.

[Request] I have a simulated user population with satisfaction scores
and two covariates, engagement_minutes and had_support_ticket, plus a
non-response mechanism where the probability of responding increases
with satisfaction. Write Python that implements inverse-probability
weighting to correct the observed survey mean, using a logistic model
of response on the covariates.

[Iterate] numpy, pandas, scikit-learn or statsmodels. Do not simulate
new data: use my existing DataFrame `after`, with columns satisfaction,
engagement_minutes, engagement_z, had_support_ticket and responded
(bool), and my existing generator `rng`. The true mean is
after["satisfaction"].mean(). Return the weighted estimate and its
bootstrap interval, and print whether the true mean is inside it.

[Mechanism Check] Explain precisely which assumption makes this
correction valid, and what happens to it when response depends on
satisfaction ITSELF rather than only on the covariates.

[Evaluate] Given my mechanism, will this correction recover the true
mean? Answer yes or no and justify it.

*Reply:*
`ipw_after.py` runs on your existing `after` and `rng`; it doesn't create any data. I couldn't run it, because your DataFrame isn't in my environment, so you'll see the numbers when you run it in your session.

**How the script works:**
- **Covariates:** The propensity model uses `engagement_z` and `had_support_ticket`. I left out `engagement_minutes` because it's a rescaled copy of `engagement_z`, so it would be perfectly collinear and give the same fitted probabilities.
- **Estimate:** It fits an unpenalized logistic regression of `responded` on those covariates, using everyone in `after`. It then computes a weighted mean of satisfaction over respondents only, with weights 1/p̂ normalized to sum to one. Non-respondents' satisfaction is never used, even though it's present in `after`.
- **Interval:** The bootstrap draws indices with your `rng`, resamples the full sample, and refits the model in each replicate. It then gives a 95% percentile interval.
- **Output:** It prints the true mean, the naive respondent mean, the IPW estimate, the interval, and `True mean inside CI?`.

**Mechanism check.** The correction is valid under one assumption: response is missing at random given the covariates. In symbols, P(responded | satisfaction, X) = P(responded | X). Among users with the same engagement and ticket status, responders and non-responders must have the same distribution of satisfaction. It also needs every user to have some chance of responding, and the logistic model to be the right shape for P(responded | X).

When response depends on satisfaction itself, that assumption breaks. The covariates explain only part of satisfaction. The remainder still drives response, so even within one covariate group, responders are more satisfied than non-responders. Weights based on X only rebalance the covariate mix; they can't correct this selection within groups. The estimate removes only the part of the bias that runs through the covariates. Meanwhile the bootstrap measures only sampling noise, so its interval shrinks around a biased value.

**Evaluation: No.** Your mechanism makes response depend directly on satisfaction. Fixing that would need covariates that explain satisfaction completely, which these two can't. I expect the IPW estimate to land between the naive mean and the true mean, and the script to print `True mean inside CI?  : False`. The more of satisfaction the covariates leave unexplained, and the more strongly response depends on it, the larger the remaining gap.
"""
IPW non-response correction on the existing DataFrame `after`,
using the existing generator `rng`. No new data is simulated.

Run this in the same session where `after` and `rng` already exist.
"""
import numpy as np
from sklearn.linear_model import LogisticRegression

# engagement_z is the standardized engagement_minutes. The two give the
# same fitted probabilities, so only one is used (both together would be
# perfectly collinear). The z-score version makes the fit numerically stabler.
COVARIATES = ["engagement_z", "had_support_ticket"]


def ipw_mean(df, covariates=COVARIATES, clip=1e-3):
    """Hajek IPW estimate of mean satisfaction from respondents only."""
    X = df[covariates].to_numpy(dtype=float)
    r = df["responded"].to_numpy(dtype=bool)
    model = LogisticRegression(C=np.inf, max_iter=1000)  # unpenalized MLE
    model.fit(X, r.astype(int))
    p_hat = np.clip(model.predict_proba(X)[:, 1], clip, 1.0)
    w = 1.0 / p_hat[r]
    y = df.loc[r, "satisfaction"].to_numpy(dtype=float)  # respondents' scores only
    return np.sum(w * y) / np.sum(w)


def bootstrap_ipw(df, rng, covariates=COVARIATES, n_boot=1000, alpha=0.05):
    """Resample the full sample (respondents + non-respondents) and refit
    the propensity model in each replicate."""
    n = len(df)
    boots = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        boots[b] = ipw_mean(df.iloc[idx], covariates)
    lo, hi = np.quantile(boots, [alpha / 2, 1 - alpha / 2])
    return (lo, hi), boots


true_mean = after["satisfaction"].mean()
naive_mean = after.loc[after["responded"], "satisfaction"].mean()
ipw_est = ipw_mean(after)
(ci_lo, ci_hi), boot_draws = bootstrap_ipw(after, rng, n_boot=1000)

print(f"Response rate         : {after['responded'].mean():.3f}")
print(f"True mean             : {true_mean:.3f}")
print(f"Naive respondent mean : {naive_mean:.3f}")
print(f"IPW estimate          : {ipw_est:.3f}")
print(f"95% bootstrap CI      : [{ci_lo:.3f}, {ci_hi:.3f}]")
print(f"True mean inside CI?  : {ci_lo <= true_mean <= ci_hi}")


Version 1 simulated its own data, so I told it to use my after DataFrame, column names and rng, and to check the true mean against the interval.

## Before you submit

- [ ] Every heading above is still here, in order, with work under it.
- [ ] Every number you report is produced by a cell above it, and every claim has a text cell.
- [ ] The verification checkpoint(s) are met before anything built on them.
- [ ] Phase 4: the AI transcript is the exchange, not a summary; both prompt versions are in it.
- [ ] Submit the Colab share link on Canvas (upload the `.ipynb` as a fallback).
- [ ] GitHub portfolio: *File › Save a copy in GitHub*, path `Problem Set 2/Econ_3916_PS2.ipynb`, with a short `README.md` in that folder.
- [ ] Run *Runtime › Restart and run all* once, top to bottom, before you hand in.